# LAO* with the windows PDB

Solves TP-MCTS's MDP (CoMDP+ states + STN) **offline and optimally** with ILAO*, guided by the windows PDB
(`windows_ilao_pdb`). The PDB is an upper bound, so LAO* can skip whole branches and still return the optimum.

Per scenario and pattern growth it reports: h(s0), V(s0), whether optimality was proven, nodes generated /
expanded, the optimal policy's size, PDB build time and search time. With `WHOLE_MDP = True` it also expands the
whole MDP (no heuristic) to show how much the PDB pruned.

Code: `comdp_plus_no_deadline/engines/lao_tp_mdp.py` (solver), `scripts/run_lao_tp_mdp.py` (one run),
`scripts/count_whole_mdp.py` (whole MDP). Results: `results/lao/`.

Boxes: **Setup** (pulls the repo on GCE) -> **Config** -> **Run** -> **Results**.

In [ ]:
# Runtime: "colab" | "gce" | "wsl" | "auto"
# - colab: clone into /content/tp_mcts (fresh each run)
# - gce:   GCE VM — /opt/tp_mcts, git pull on Setup (see gcp/README.md)
# - wsl:   use local checkout on Linux (WSL); no clone
# - auto:  Colab → GCE marker → WSL path → cwd
import os
import shutil
import subprocess
import sys
from pathlib import Path

RUN_ENV = "auto"  # "colab" | "gce" | "wsl" | "auto" (auto detects the GCE VM / WSL / Colab)

# WSL: Windows path C:\Users\...\TP_MCTS → /mnt/c/Users/.../TP_MCTS
# Override with env TP_MCTS_REPO_ROOT if your clone lives elsewhere.
WSL_REPO_ROOT = Path(
    os.environ.get(
        "TP_MCTS_REPO_ROOT",
        "/mnt/c/Users/eliezer/Documents/hw2_exam/TP_MCTS",
    )
)

COLAB_REPO_DIR = "/content/tp_mcts"
GCE_REPO_DIR = os.environ.get("TP_MCTS_GCE_REPO", "/opt/tp_mcts")
GCE_MARKER = Path(GCE_REPO_DIR) / ".tp_mcts_gce"
REPO_URL = "https://github.com/eliezerRevach/tp_mcts.git"
# master on GCE; set to a feature branch name when testing unpushed work
REPO_BRANCH = "master"  # None = default branch on GitHub

PIP_DEPS = ["dill", "numpy", "pandas", "openpyxl", "networkx"]


def _is_tp_mcts_root(path: Path) -> bool:
    return (
        (path / "scripts" / "run_mcts_heuristic_comparison.py").is_file()
        and (path / "unified_planning" / "run_domain.py").is_file()
        and (path / "experiments.ipynb").is_file()
    )


def _find_repo_from_cwd() -> Path | None:
    start = Path.cwd().resolve()
    candidates = [p for p in [start, *start.parents] if _is_tp_mcts_root(p)]
    return candidates[-1] if candidates else None


def _is_gce_host() -> bool:
    return GCE_MARKER.is_file()


def _resolve_run_env() -> str:
    if RUN_ENV not in ("colab", "gce", "wsl", "auto"):
        raise ValueError(f"RUN_ENV must be colab, gce, wsl, or auto (got {RUN_ENV!r})")
    if RUN_ENV != "auto":
        return RUN_ENV
    if os.path.isdir("/content"):
        return "colab"
    if _is_gce_host():
        return "gce"
    if sys.platform.startswith("linux") and _is_tp_mcts_root(WSL_REPO_ROOT):
        return "wsl"
    if _find_repo_from_cwd() is not None:
        return "wsl"
    if sys.platform == "win32":
        return "windows"
    return "wsl"


def _git_sync_repo(repo: Path, *, fresh_clone_dir: str | None = None) -> None:
    """Clone into fresh_clone_dir, or pull existing repo at repo."""
    if fresh_clone_dir is not None:
        if os.path.isdir(fresh_clone_dir):
            shutil.rmtree(fresh_clone_dir, ignore_errors=True)
        parent = str(Path(fresh_clone_dir).parent)
        os.makedirs(parent, exist_ok=True)
        clone_cmd = ["git", "clone", REPO_URL, fresh_clone_dir]
        if REPO_BRANCH:
            clone_cmd = ["git", "clone", "--branch", str(REPO_BRANCH), REPO_URL, fresh_clone_dir]
        subprocess.check_call(clone_cmd)
        return
    if not (repo / ".git").is_dir():
        raise RuntimeError(f"Not a git repo: {repo}")
    subprocess.check_call(["git", "-C", str(repo), "fetch", "--all", "--prune"])
    if REPO_BRANCH:
        subprocess.check_call(["git", "-C", str(repo), "checkout", str(REPO_BRANCH)])
    subprocess.check_call(["git", "-C", str(repo), "pull", "--ff-only"])


def _pip_install() -> None:
    py = sys.executable
    gce_venv = Path(GCE_REPO_DIR) / ".venv" / "bin" / "python"
    if RESOLVED_RUN_ENV == "gce" and gce_venv.is_file():
        py = str(gce_venv)
    subprocess.check_call([py, "-m", "pip", "-q", "install", *PIP_DEPS])


def _python_cmd() -> str:
    """Interpreter for experiment subprocesses (same env as notebook kernel)."""
    return sys.executable


def _print_git_head(repo: Path) -> None:
    rev = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True
    ).strip()
    print("Git HEAD:", rev)


RESOLVED_RUN_ENV = _resolve_run_env()
os.environ["TP_MCTS_RUN_ENV"] = RESOLVED_RUN_ENV

if RESOLVED_RUN_ENV == "windows":
    raise RuntimeError(
        "Native Windows Python is not supported for these experiments (Linux required).\n"
        "  1. Install WSL and open this repo under /mnt/c/.../TP_MCTS\n"
        "  2. In VS Code/Cursor: Select Kernel → WSL Python\n"
        "  3. Re-run this cell with RUN_ENV = 'wsl' and WSL_REPO_ROOT set correctly"
    )

if RESOLVED_RUN_ENV == "colab":
    if RUN_ENV == "colab" and not os.path.isdir("/content"):
        raise RuntimeError("RUN_ENV='colab' but /content is missing — not on Colab?")
    if not COLAB_REPO_DIR.startswith("/content/"):
        raise RuntimeError(f"Refusing to delete unexpected path: {COLAB_REPO_DIR}")
    os.chdir("/content")
    _git_sync_repo(Path(COLAB_REPO_DIR), fresh_clone_dir=COLAB_REPO_DIR)
    repo = Path(COLAB_REPO_DIR)
    os.chdir(repo)
    _print_git_head(repo)
    if REPO_BRANCH:
        print("branch=", REPO_BRANCH)
    _pip_install()
    os.environ["TP_MCTS_REPO_ROOT"] = str(repo.resolve())
    os.environ["TP_MCTS_PYTHON"] = sys.executable
    print("RUN_ENV: colab")
    print("Ready:", os.getcwd())

elif RESOLVED_RUN_ENV == "gce":
    if not sys.platform.startswith("linux"):
        raise RuntimeError(
            "RUN_ENV='gce' must run ON the Compute Engine VM (Linux), not on Windows.\n"
            f"  Your kernel is local Python on {sys.platform!r}.\n"
            "  Fix:\n"
            "    1. Cursor → Remote-SSH → tp-mcts-64cpu.us-east1-d.integral-bliss-498316-j2\n"
            "    2. Open folder /opt/tp_mcts and this notebook there\n"
            "    3. Select Kernel → Python 3 or 'TP-MCTS' (/opt/tp_mcts/.venv/bin/python)\n"
            "    4. Re-run Setup with RUN_ENV = 'gce'\n"
            "  To run on this PC instead: use RUN_ENV = 'wsl' and a WSL Jupyter kernel."
        )
    repo = Path(GCE_REPO_DIR)
    if not _is_tp_mcts_root(repo):
        raise RuntimeError(
            "GCE repo not ready. Run gcp/create-vm.sh or on the VM:\n"
            f"  sudo bash {GCE_REPO_DIR}/gcp/bootstrap-vm.sh\n"
            f"  (expected marker {GCE_MARKER})"
        )
    _git_sync_repo(repo)
    os.chdir(repo)
    _print_git_head(repo)
    if REPO_BRANCH:
        print("branch=", REPO_BRANCH)
    _pip_install()
    os.environ["TP_MCTS_REPO_ROOT"] = str(repo.resolve())
    venv_py = repo / ".venv" / "bin" / "python"
    if venv_py.is_file():
        os.environ["TP_MCTS_PYTHON"] = str(venv_py)
    print("RUN_ENV: gce (Compute Engine — results persist in results/)")
    print("Kernel should use:", os.environ.get("TP_MCTS_PYTHON", sys.executable))
    print("Ready:", os.getcwd())
    print("Copy results to PC: .\\gcp\\fetch-results.ps1  (see gcp/README.md)")

elif RESOLVED_RUN_ENV == "wsl":
    if not sys.platform.startswith("linux"):
        raise RuntimeError(
            f"RUN_ENV='wsl' requires Linux (got {sys.platform}). Use a WSL Jupyter kernel."
        )
    repo = WSL_REPO_ROOT.resolve() if _is_tp_mcts_root(WSL_REPO_ROOT) else _find_repo_from_cwd()
    if repo is None:
        raise RuntimeError(
            "Could not find TP-MCTS repo.\n"
            f"  WSL_REPO_ROOT={WSL_REPO_ROOT} (missing scripts/ or unified_planning/)\n"
            "  Set WSL_REPO_ROOT to your clone, e.g. /mnt/c/Users/you/.../TP_MCTS\n"
            "  Or cd into the repo before running Setup."
        )
    os.chdir(repo)
    os.environ["TP_MCTS_REPO_ROOT"] = str(repo)
    os.environ["TP_MCTS_PYTHON"] = sys.executable
    _print_git_head(repo)
    _pip_install()
    print("RUN_ENV: wsl (local Linux — no git clone)")
    print("Ready:", os.getcwd())

else:
    raise RuntimeError(f"Unhandled RUN_ENV resolution: {RESOLVED_RUN_ENV}")

## Config

In [ ]:
# ── LAO* config ──────────────────────────────────────────────────────────────
import itertools
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("TP_MCTS_REPO_ROOT", Path.cwd())).resolve()
PYTHON = os.environ.get("TP_MCTS_PYTHON", sys.executable)

# ── Scenarios: one LAO* run per (objects, garbage, deadline) x pattern growth ──
DOMAIN    = "nasa_rover"   # nasa_rover | stuck_car | prob_conc | machine_shop | ... (scripts/run_lao_tp_mdp.py)
OBJECTS   = [2]            # object_amount (domains with objects; others ignore it -> keep [1])
GARBAGE   = [0]            # garbage_amount (prob_conc, simple, hosting; others ignore it -> keep [0])
DEADLINES = [25]

# ── What to run ───────────────────────────────────────────────────────────────
PATTERN_GROWTHS = ["pairs", "llm_by_hand_nasa_two"]  # how the PDB picks its facts, one LAO* run each:
#                  "pairs" (automatic: goals + one shared resource each) | "llm_by_hand_nasa_two" (facts picked by
#                  hand, nasa_rover obj 2 ONLY) | "cegar" | "static" | "logic" (see experiments.ipynb)
CANONICAL_STARTS = [False]  # [False, True] runs both: True = two independent starts in a row only in name order
#                            (fewer start orderings; exact on 25/25 stuck_car / prob_conc checks, not proven)
PREBUILD_TABLES = True     # build every scenario's PDB tables first (no memory cap), then run the jobs from the cache
PREBUILD_PARALLEL = False  # True: all builds at once (fast, needs the memory of all of them together)
WHOLE_MDP_MAX_DEADLINE = 0 # whole MDP only for deadlines <= this (0 = every scenario)
WHOLE_MDP_MEMORY_GB    = 0 # memory cap per whole-MDP job (0 = MEMORY_LIMIT_GB)
WHOLE_MDP       = True     # also expand the whole MDP (no heuristic, no pruning) -> what the PDB saved
REJECT_AFTER    = 1800     # seconds per run, PDB build + search together (whole MDP: its own budget); 0 = none
VALIDATE_EXACT  = False    # after solving: execute the policy over every outcome with times fixed as executed
VALIDATE_RUNS   = 0        # after solving: this many sampled executions (mdp.step), checked step by step
PARALLEL        = True     # run every job at the same time (each job is one process)
MEMORY_LIMIT_GB = 0        # per job (Linux): over it the job stops and reports what it reached; 0 = none
DIAGNOSTICS     = True     # per-depth report: STN-rejected actions, ties, gradient (Q spread), h-pruned options,
#                            tightness h0 vs V* (finished runs), lower bound on V(s0) every DIAGNOSTICS_INTERVAL s
DIAGNOSTICS_INTERVAL = 600
DIAGNOSTICS_TIPS = 5       # per sample: the tips carrying most of V(s0) -- facts, time left, h per pattern, value
#                            one expansion later (shows where h is too optimistic); 0 = off
OUT_DIR         = REPO_ROOT / "results" / "lao"

# ── Windows PDB (same meaning as the WILAO_* settings in experiments.ipynb) ────
WILAO_BLUR           = 3       # window grid eps: lo down / hi up to eps (fewer states, still an upper bound); 0 = off
WILAO_PATTERN_GOALS  = 1       # goal facts per pattern; 0 = all goals in one joint pattern
WILAO_PATTERN_FACTS  = 10      # facts per pattern
WILAO_GROUPING       = True    # h = product over independent goal groups of (min over the group's patterns)
WILAO_MISS           = "one"   # a lookup that misses the table counts as 1 (upper bound)
WILAO_EXACT_STATICS  = True    # facts no action changes keep their initial value in every pattern
WILAO_NO_STN_PASS    = False
WILAO_OFFLINE_EXTEND = True
WILAO_TABLE_CACHE    = str(REPO_ROOT / "results" / "wilao_cache")  # build each table once per setting, reuse it

PDB_ENV = {
    "TP_MCTS_WILAO_OFFLINE_SECONDS": "0",             # LAO* needs the full tables: no offline limit
    "TP_MCTS_WILAO_OFFLINE_EXTEND": "1" if WILAO_OFFLINE_EXTEND else "0",
    "TP_MCTS_WILAO_PATTERN_GOALS": str(WILAO_PATTERN_GOALS),
    "TP_MCTS_WILAO_PATTERN_FACTS": str(WILAO_PATTERN_FACTS),
    "TP_MCTS_WILAO_MISS": WILAO_MISS,
    "TP_MCTS_WILAO_AGG": "same",                      # = min inside a group for pairs / hand patterns
    "TP_MCTS_WILAO_GROUPING": "1" if WILAO_GROUPING else "0",
    "TP_MCTS_WILAO_BLUR": str(WILAO_BLUR),
    "TP_MCTS_WILAO_EXACT_STATICS": "1" if WILAO_EXACT_STATICS else "0",
    "TP_MCTS_WILAO_NO_STN_PASS": "1" if WILAO_NO_STN_PASS else "0",
    "TP_MCTS_WILAO_TABLE_CACHE": WILAO_TABLE_CACHE,
    "TP_MCTS_WILAO_REPORT": "0",
}
SCENARIOS = list(itertools.product(OBJECTS, GARBAGE, DEADLINES))
print(f"LAO* config: domain={DOMAIN}  scenarios (objects, garbage, deadline)={SCENARIOS}")
print(f"  growths={PATTERN_GROWTHS}  whole_mdp={WHOLE_MDP}  reject_after={REJECT_AFTER}s  parallel={PARALLEL}")
print(f"  memory/job={MEMORY_LIMIT_GB or 'none'} GB  diagnostics={DIAGNOSTICS}  canonical_starts={CANONICAL_STARTS}")
print(f"  PDB: blur={WILAO_BLUR} grouping={WILAO_GROUPING} facts={WILAO_PATTERN_FACTS} miss={WILAO_MISS} cache={WILAO_TABLE_CACHE}")
print(f"  results -> {OUT_DIR}")

## Run

In [ ]:
# ── Run: every job as its own process; a status line every 60 s ────────────────
import subprocess
import time

OUT_DIR.mkdir(parents=True, exist_ok=True)
base_env = os.environ.copy()
base_env.update(PDB_ENV)
base_env["PYTHONPATH"] = str(REPO_ROOT)

jobs = []                                            # (name, cmd, extra env, json path, log path)
for obj, garbage, deadline in SCENARIOS:
    tag = f"{DOMAIN}_o{obj}_g{garbage}_d{deadline}"
    scenario = ["--domain", DOMAIN, "--object_amount", str(obj), "--garbage_amount", str(garbage),
                "--deadline", str(deadline)]
    for growth, canon in itertools.product(PATTERN_GROWTHS, CANONICAL_STARTS):
        out = OUT_DIR / f"lao_{tag}_{growth}{'_canon' if canon else ''}.json"
        cmd = [PYTHON, "-u", str(REPO_ROOT / "scripts" / "run_lao_tp_mdp.py"), *scenario, "--heuristic", "pdb",
               "--reject_after", str(REJECT_AFTER), "--progress", "--json", str(out)]
        if canon:
            cmd.append("--canonical_starts")
        if VALIDATE_EXACT:
            cmd.append("--validate_exact")
        if VALIDATE_RUNS:
            cmd += ["--validate_runs", str(VALIDATE_RUNS)]
        if DIAGNOSTICS:
            cmd += ["--diagnostics", "--diagnostics_interval", str(DIAGNOSTICS_INTERVAL),
                    "--diagnostics_tips", str(DIAGNOSTICS_TIPS)]
        jobs.append((f"LAO* {tag} {growth}{' canon' if canon else ''}", cmd, {"TP_MCTS_WILAO_PATTERN_GROWTH": growth}, out,
                     out.with_suffix(".log")))
    if WHOLE_MDP and (not WHOLE_MDP_MAX_DEADLINE or deadline <= WHOLE_MDP_MAX_DEADLINE):
        out = OUT_DIR / f"whole_{tag}.json"
        cmd = [PYTHON, "-u", str(REPO_ROOT / "scripts" / "count_whole_mdp.py"), *scenario,
               "--budget", str(REJECT_AFTER), "--json", str(out)]
        jobs.append((f"whole MDP {tag}", cmd, {"_MEMORY_GB": str(WHOLE_MDP_MEMORY_GB or MEMORY_LIMIT_GB)}, out,
                     out.with_suffix(".log")))

for _name, _cmd, _env, out, _log in jobs:            # no stale result from an earlier run
    if out.exists():
        out.unlink()


def _memory_cap(gb):
    """A function run in the child before it starts: cap its address space (Linux), so a run over the
    limit gets a MemoryError and reports, instead of the machine swapping or killing it."""
    def cap_it():
        if gb and sys.platform.startswith("linux"):
            import resource
            cap = int(float(gb) * 1024 ** 3)
            resource.setrlimit(resource.RLIMIT_AS, (cap, cap))
    return cap_it


if PREBUILD_TABLES:                                  # no memory cap, into the table cache
    builds = []
    t0 = time.time()
    for obj, garbage, deadline in SCENARIOS:
        for growth in PATTERN_GROWTHS:
            tag = f"{DOMAIN}_o{obj}_g{garbage}_d{deadline}_{growth}"
            fh = open(OUT_DIR / f"build_{tag}.log", "w")
            proc = subprocess.Popen([PYTHON, "-u", str(REPO_ROOT / "scripts" / "run_lao_tp_mdp.py"),
                                     "--domain", DOMAIN, "--object_amount", str(obj), "--garbage_amount",
                                     str(garbage), "--deadline", str(deadline), "--heuristic", "pdb",
                                     "--build_only", "--reject_after", "0", "--json",
                                     str(OUT_DIR / f"build_{tag}.json")],
                                    stdout=fh, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT),
                                    env=dict(base_env, TP_MCTS_WILAO_PATTERN_GROWTH=growth))
            if not PREBUILD_PARALLEL:
                proc.wait()
                fh.close()
                print(f"built tables {tag}: exit {proc.returncode} in {time.time() - t0:.0f} s", flush=True)
            else:
                builds.append((tag, proc, fh))
    for tag, proc, fh in builds:
        proc.wait()
        fh.close()
        print(f"built tables {tag}: exit {proc.returncode} ({time.time() - t0:.0f} s since the first build)",
              flush=True)


def _last_line(path):
    try:
        lines = [l for l in path.read_text(errors="replace").splitlines() if l.startswith("[")]
        return lines[-1][:110] if lines else ""
    except OSError:
        return ""


started = time.time()
running = []
for name, cmd, extra, out, log in jobs:
    extra = dict(extra)
    gb = extra.pop("_MEMORY_GB", MEMORY_LIMIT_GB)
    env = dict(base_env, **extra)
    fh = open(log, "w")
    print(f"start {name}: {' '.join(cmd[2:])}", flush=True)
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT), env=env,
                            preexec_fn=_memory_cap(gb) if sys.platform.startswith("linux") else None)
    running.append((name, proc, fh, log))
    if not PARALLEL:
        proc.wait()
last_report = time.time()
while any(p.poll() is None for _n, p, _f, _l in running):
    time.sleep(5)
    if time.time() - last_report < 60:
        continue
    last_report = time.time()
    mins = (time.time() - started) / 60
    for name, proc, _fh, log in running:
        state = "running" if proc.poll() is None else f"done (exit {proc.returncode})"
        print(f"[{mins:5.1f} min] {name:<45} {state:<16} {_last_line(log)}", flush=True)
for name, proc, fh, log in running:
    fh.close()
    print(f"{name}: exit {proc.returncode}  log {log}", flush=True)
print(f"all jobs done in {(time.time() - started) / 60:.1f} min")

## Results

In [ ]:
# ── Results: one row per LAO* run, with the whole MDP next to it ───────────────
import json
import pandas as pd


def _load(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return None


rows = []
for obj, garbage, deadline in SCENARIOS:
    tag = f"{DOMAIN}_o{obj}_g{garbage}_d{deadline}"
    whole = _load(OUT_DIR / f"whole_{tag}.json") if WHOLE_MDP else None
    for growth, canon in itertools.product(PATTERN_GROWTHS, CANONICAL_STARTS):
        r = _load(OUT_DIR / f"lao_{tag}_{growth}{'_canon' if canon else ''}.json") or {}
        done = r.get("rejected") is False
        row = {
            "domain": DOMAIN, "objects": obj, "garbage": garbage, "deadline": deadline, "growth": growth,
            "canonical": canon,
            "status": "optimal" if done and r.get("optimal") else (r.get("reason") or "missing/crashed"),
            "peak_rss_gb": r.get("peak_rss_gb"),
            "h(s0)": r.get("h(s0)"),
            "V(s0)": r.get("V(s0)") if done else r.get("V(s0) so far (upper bound)"),
            "generated": r.get("nodes_generated"), "expanded": r.get("nodes_expanded"),
            "policy_nodes": (r.get("policy") or {}).get("total"),
            "pdb_build_s": r.get("pdb_build_seconds"), "search_s": r.get("search_seconds"),
            "table_states": (r.get("pdb") or {}).get("table_states"),
        }
        if whole:
            exact = whole.get("finished")
            row["whole_nodes"] = ("" if exact else ">= ") + f"{whole['whole_mdp_nodes']:,}"
            row["whole_expanded"] = ("" if exact else ">= ") + f"{whole['whole_mdp_expanded']:,}"
            if done and row["generated"]:
                saved_g = 100 * (1 - row["generated"] / whole["whole_mdp_nodes"])
                saved_e = 100 * (1 - row["expanded"] / max(1, whole["whole_mdp_expanded"]))
                row["never_created"] = ("" if exact else ">= ") + f"{saved_g:.1f}%"
                row["expansions_avoided"] = ("" if exact else ">= ") + f"{saved_e:.2f}%"
            else:
                row["never_created"] = row["expansions_avoided"] = "n/a (LAO* not finished)"
        if r.get("execution_exact"):
            row["executed_value"] = r["execution_exact"].get("executed_value")
            row["violations"] = r["execution_exact"].get("violation_mass")
        rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "lao_summary.csv", index=False)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
print(summary.to_string(index=False))
print(f"\nsaved {OUT_DIR / 'lao_summary.csv'}")
print("generated = (state, STN) nodes LAO* created; whole_nodes = every node of the MDP (no heuristic);")
print("'>=' = that run hit its limit, so the count is a lower bound; V(s0) of an unfinished run = upper bound so far")

# ── Diagnostics: what happened inside each run ───────────────────────────────
if DIAGNOSTICS:
    for obj, garbage, deadline in SCENARIOS:
        tag = f"{DOMAIN}_o{obj}_g{garbage}_d{deadline}"
        for growth, canon in itertools.product(PATTERN_GROWTHS, CANONICAL_STARTS):
            r = _load(OUT_DIR / f"lao_{tag}_{growth}{'_canon' if canon else ''}.json") or {}
            d = r.get("diagnostics")
            if not d:
                continue
            print()
            print(f"=== diagnostics: {tag} {growth}{' canonical' if canon else ''}  ({r.get('reason') or ('optimal' if r.get('optimal') else '')}) ===")
            print(f"h-pruned options: {d.get('h_pruned_options_pct', 'n/a')}%   zero-time cycles: {d.get('cycles')}"
                  f"   folded self-loops: {d.get('folded')}   canonical skips: {d.get('canonical_skipped')}")
            t = d.get("tightness")
            if isinstance(t, dict):
                print(f"tightness on the optimal policy: mean h0-V* = {t['gap_mean']}  max = {t['gap_max']}"
                      f"  ({t['policy_decision_nodes']} decision nodes, admissibility violations: {t['violations']})")
            elif t:
                print(f"tightness: {t}")
            prog = pd.DataFrame([{k: v for k, v in p.items() if k != "top_tips"} for p in d["progress"]])
            if not prog.empty:
                print("bounds on V(s0) and the best policy's size over time:")
                print(prog.to_string(index=False))
            sampled = [p for p in d["progress"] if p.get("top_tips")]
            if sampled:
                last = sampled[-1]
                print(f"tips carrying most of V(s0) at {last['seconds']} s (upper bound {last['upper_bound']}):")
                for tip in last["top_tips"]:
                    binding = [f"{pt['goals']}={pt['value']}" for pt in tip.get("patterns", []) if pt.get("binding")]
                    print(f"  share {tip['share_of_upper']:.3f}  reach {tip['reach']:.4f}  h {tip['value']:.4f}"
                          f" -> after one expansion {tip.get('after_expand')}  (dead: {tip.get('after_expand_dead')})"
                          f"  depth {tip['depth']}  time left {tip['time_left']}")
                    print(f"     running {tip['running']}")
                    print(f"     gained {tip['gained']}   lost {tip['lost']}")
                    print(f"     binding pattern(s): {binding}")
            for title, key in (("time left", "by_time_left"), ("running actions", "by_running_actions"),
                               ("goals done", "by_goals_done"), ("action that created the node", "by_creating_action"),
                               ("how much each expanded node's h has dropped so far, per depth", "h_drop_by_depth")):
                if d.get(key):
                    print(f"nodes by {title}:")
                    print(pd.DataFrame(d[key]).to_string(index=False))
            if d.get("alternative_gap_to_best"):
                print("alternatives vs the best option at expansion (Q_best - Q_alt):",
                      {k: f"{v['count']} ({v['pct']}%)" for k, v in d["alternative_gap_to_best"].items()})
            if d.get("work_outside_best_policy"):
                print("work outside the best policy:", d["work_outside_best_policy"])
            if isinstance(t, dict) and t.get("loose_nodes_by_weakest_group"):
                print("optimal-policy nodes with h0 - V* > 0.01, by the PDB group that was weakest there:")
                print(pd.DataFrame(t["loose_nodes_by_weakest_group"]).to_string(index=False))
            dom = d.get("dominance")
            if dom:
                print(f"dominance: {dom['dominated']:,} of {dom['nodes']:,} nodes are dominated (same facts, looser STN"
                      f" elsewhere) = {dom['dominated_pct']}%;  {dom['distinct_facts']:,} distinct fact sets;"
                      f"  scanned {dom['scanned_groups']:,} of {dom['groups_with_2_or_more']:,} groups"
                      f" ({dom['dominated_pct_of_scanned']}% of the scanned nodes dominated)")
                print(pd.DataFrame(dom["largest_groups"]).to_string(index=False))
            if d.get("by_root_option"):
                print("where the nodes are, per first move from the root:")
                print(pd.DataFrame(d["by_root_option"]).to_string(index=False))
            if d.get("main_path"):
                print("main path (best action, most likely outcome):")
                print(pd.DataFrame(d["main_path"]).to_string(index=False))
            for grp in d.get("same_facts_groups") or []:
                print(f"same facts at time {grp['time']}: {grp['size']} nodes, depths {grp['depths']},"
                      f" under root options {grp['root_options']}")
                for m in grp["members"]:
                    print(f"   depth {m['depth']} value {m['value']} expanded {m['expanded']}")
                    print(f"      path: {' > '.join(m['path'])}")
                    print(f"      stn:  {m['signature']}")
            print("per depth:")
            print(pd.DataFrame(d["by_depth"]).to_string(index=False))
            if isinstance(t, dict) and t.get("by_depth"):
                print("tightness per depth:")
                print(pd.DataFrame(t["by_depth"]).to_string(index=False))
    print()
    print("legend: stn_rejected_pct = legal actions the STN rejects; ties_within_x = options per node within x of the"
          " best Q_h (an upper bound cannot cut them); q_spread = best - worst Q_h (the gradient);"
          " h_pruned_pct = options never chosen and never expanded below; merged_pct = children that already existed;"
          " nodes_per_facts_time = nodes per (facts, time) group: > 1 = same facts at the same time, only the STN differs")